# 04 – Modellalapú kockázatelemzés: rendszermodelltől a hibafáig

**4. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- rendszermodell és kockázati modell szerepének elkülönítése
- simplified SysML-derived exchange representation értelmezése
- strukturális és referencia-validáció
- determinista model-to-FTA transzformáció és traceability

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Miért model-based?

A kézzel karbantartott FTA gyorsan inkonzisztenssé válhat, ha a rendszerstruktúra változik. A cél ezért explicit kapcsolat a **rendszerleírás** és a **kockázati modell** között.

Ebben a tanegységben nem natív SysML parserrel dolgozunk. Egy **simplified SysML-derived YAML/JSON exchange representation** szerepel: komponensek, funkciók, failure mode-ok, hazard logic és kapcsolatok. Ez oktatási reprezentáció, nem a teljes SysML szemantika.


## 2. Minimális rendszermodell

Két safety funkció: LSHH-101 indítja a shutdown-t, XV-101 zárja a betáplálást. Mindkettő failure-on-demand eseménnyel rendelkezik. A hazard logic azt mondja: bármelyik hiba elég az automatikus leállítás sikertelenségéhez.


In [ ]:
model={
 "components":[
  {"id":"LSHH101","type":"high_high_switch","function":"initiate_shutdown","failure_modes":[{"id":"LSHH101_FOD","mode":"fail_on_demand","probability":.01}]},
  {"id":"XV101","type":"shutdown_valve","function":"stop_inflow","failure_modes":[{"id":"XV101_FOD","mode":"fail_on_demand","probability":.005}]}
 ],
 "hazards":[{"id":"AUTO_FAIL","logic":{"gate":"OR","inputs":[{"event":"LSHH101_FOD"},{"event":"XV101_FOD"}]}}]
}
model


## 3. YAML csak csereformátum

A tudományos tartalom nem a YAML szintaxis, hanem a mezők definíciója, a validálási szabályok és a transzformáció. A formátum később natív SysML/AADL/Simulink adapterrel cserélhető.


In [ ]:
import yaml
print(yaml.safe_dump(model,sort_keys=False,allow_unicode=True))


## 4. Validáció generálás előtt

Legalább: unique ID, minden hazardnak logic, minden `event` referencia létező failure mode-ra mutat, és csak támogatott gate szerepel.


In [ ]:
def catalog(m):
    return {fm["id"]:{**fm,"component":c["id"]} for c in m.get("components",[]) for fm in c.get("failure_modes",[])}

def validate(m):
    errors=[]; ids=[c.get("id") for c in m.get("components",[])]
    if len(ids)!=len(set(ids)): errors.append("duplicate component id")
    cat=catalog(m)
    def check(x):
        if "event" in x:
            if x["event"] not in cat: errors.append("unknown event: "+x["event"])
            return
        if str(x.get("gate","")).upper() not in {"AND","OR"}: errors.append("unsupported gate")
        for y in x.get("inputs",[]): check(y)
    for h in m.get("hazards",[]):
        if "logic" not in h: errors.append("hazard without logic")
        else: check(h["logic"])
    return errors
print(validate(model))


In [ ]:
from copy import deepcopy
bad=deepcopy(model)
bad["hazards"][0]["logic"]["inputs"].append({"event":"DOES_NOT_EXIST"})
print(validate(bad))


## 5. Determinisztikus transzformáció: hazard logic → FTA

Szabály: event → BASIC node; gate → FTA gate; a hazard ID → top event. A transzformáció nem LLM-döntés, hanem explicit algoritmus.


In [ ]:
def model_to_fta(m,hazard_id):
    errs=validate(m)
    if errs: raise ValueError("; ".join(errs))
    h=next(x for x in m["hazards"] if x["id"]==hazard_id); cat=catalog(m); nodes={}; k=0
    def conv(expr,preferred=None):
        nonlocal k
        if "event" in expr:
            e=expr["event"]; nodes[e]={"type":"BASIC","probability":float(cat[e]["probability"]),"source_component":cat[e]["component"]}; return e
        k+=1; gid=preferred or f"G{k:02d}"; children=[conv(x) for x in expr["inputs"]]
        nodes[gid]={"type":expr["gate"].upper(),"children":children,"source":"hazard_logic"}; return gid
    top=conv(h["logic"],hazard_id)
    return {"top_event":top,"nodes":nodes}
fta=model_to_fta(model,"AUTO_FAIL")
fta


## 6. Teljes TK-101 modell betöltése

A közös `system_model.yaml` ugyanilyen elven tartalmazza az oktatási rendszerkomponenseket és hazard logic-ot.


In [ ]:
path=ROOT/'data'/'pundit_case'/'system_model.yaml'
with open(path,encoding='utf-8') as f: tk_model=yaml.safe_load(f)
print(tk_model.get('metadata',{})); print('validation:',validate(tk_model))


In [ ]:
tk_fta=model_to_fta(tk_model,'OVERFLOW')
trace=pd.DataFrame([{"fta_node":nid,"type":n['type'],"source_component":n.get('source_component')} for nid,n in tk_fta['nodes'].items()])
trace


## 7. Mit nem állítunk?

Nem olvasunk natív SysML v2 repository/API modellt; nem értelmezzük a teljes SysML szemantikát; és nem állítjuk, hogy pusztán komponenskapcsolatokból automatikusan helyes failure propagation származik. A formális transzformáció minősége a forrásmodell és a szabályok minőségétől függ.


In [ ]:
from safetycourse.model_io import load_yaml,validate_system_model,generate_fault_tree
lib_model=load_yaml(path); lib_tree=generate_fault_tree(lib_model,'OVERFLOW')
print(validate_system_model(lib_model)); print(lib_tree['top_event'],len(lib_tree['nodes']))


## Próbáld ki!

1. Tegyél nem létező event referenciát a modellbe és ellenőrizd a validátort.
2. Adj új komponens failure mode-ot és használd hazard logicban.
3. Bővítsd a traceability táblát `hazard_id` és `gate_path` mezővel.
4. Írd le, milyen információ hiányzik egy valódi SysML-v2 → FTA adapterhez.


## Összefoglalás

A modellalapú kockázatelemzés lényege a reprodukálható **model → validation → transformation → risk model → traceability** lánc. A következő tanegységben a statikus valószínűségi modellt Bayes-hálóval evidenciára érzékennyé tesszük.
